# Policy evolution
Weight norms, log-std decay, observation normalization, and weight heatmaps at key rounds.

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

H5_PATH = "campaign_analysis.h5"
plt.rcParams.update({'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3})

f = h5py.File(H5_PATH, 'r')
p = f['policy']
rounds = p['rounds'][:]
print(f"Policy rounds: {rounds[0]}–{rounds[-1]}  ({len(rounds)} total)")

In [ ]:
# ---- exploration collapse: exp(log_std) vs round ----
ls = p['log_std'][:]  # (R, 2)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(rounds, np.exp(ls[:, 0]), label='aP (pressure)')
ax.plot(rounds, np.exp(ls[:, 1]), label='aS (shear)')
ax.set_xlabel('round'); ax.set_ylabel('action std σ = exp(log_std)')
ax.set_title('Exploration collapse')
ax.legend()
fig.tight_layout()
fig.savefig('policy_logstd.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- weight norms vs round ----
fig, ax = plt.subplots(figsize=(8, 3))
for key, label in [('W0_norm', 'W0 (input→h0)'), ('W1_norm', 'W1 (h0→h1)'), ('Wmu_norm', 'Wμ (h1→action)')]:
    ax.plot(rounds, p[key][:], label=label)
ax.set_xlabel('round'); ax.set_ylabel('Frobenius norm')
ax.set_title('Weight matrix norms')
ax.legend()
fig.tight_layout()
fig.savefig('policy_weight_norms.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- observation normalisation evolution ----
OBS_NAMES = ["fInf", "P_int", "Egamma", "gamma", "phi", "z", "maxOv", "t", "prev_aP", "prev_aS"]
obs_mean = p['obs_mean'][:]  # (R, 10)
obs_std  = p['obs_std'][:]   # (R, 10)

fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharey=False)
for i, (ax, name) in enumerate(zip(axes.ravel(), OBS_NAMES)):
    mu = obs_mean[:, i]; sg = obs_std[:, i]
    ax.fill_between(rounds, mu - sg, mu + sg, alpha=0.2)
    ax.plot(rounds, mu, lw=1.2)
    ax.set_title(name, fontsize=8)
    ax.set_xlabel('round', fontsize=7)
fig.suptitle('Obs normalisation: mean ± std per dimension', fontsize=11)
fig.tight_layout()
fig.savefig('policy_obs_norm.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ---- weight heatmaps at 4 key rounds ----
n = len(rounds)
key_indices = [0, n//4, n//2, n-1]
key_rounds  = [rounds[i] for i in key_indices]

W0  = p['W0'][:]   # (R, h0, 10)
W1  = p['W1'][:]   # (R, h1, h0)
Wmu = p['Wmu'][:]  # (R, 2, h1)

matrices = [('W0 (in→h0)', W0), ('W1 (h0→h1)', W1), ('Wμ (h1→act)', Wmu)]

fig, axes = plt.subplots(len(matrices), 4, figsize=(14, len(matrices)*3))
for row, (name, W) in enumerate(matrices):
    vmax = np.nanpercentile(np.abs(W), 99)
    for col, (idx, rnd) in enumerate(zip(key_indices, key_rounds)):
        ax = axes[row, col]
        im = ax.imshow(W[idx], aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        ax.set_title(f'{name}\nround {rnd}', fontsize=8)
        ax.set_xlabel('input', fontsize=7); ax.set_ylabel('output', fontsize=7)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Weight matrices at key rounds', fontsize=11)
fig.tight_layout()
fig.savefig('policy_weight_heatmaps.pdf', bbox_inches='tight')
plt.show()